# DEH 30-Day PySpark Challenge
## Day 2 — RDDs & SparkContext

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

## Step 1 — Install PySpark

In [ ]:
!pip install pyspark --quiet

## Step 2 — AWS Credentials & Bucket

In [ ]:
import os

os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'

BUCKET = 'deh-pyspark-challenge-your-account-id'

print('Credentials set.')

## Step 3 — Create SparkSession & SparkContext

In [ ]:
import pyspark
print(f"PySpark version: {pyspark.__version__}")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (
    SparkSession
    .builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)

# Access SparkContext from SparkSession
sc = spark.sparkContext

print(f"Spark version : {spark.version}")
print(f"Master        : {sc.master}")

## Step 4 — Creating RDDs with parallelize()

In [ ]:
# Create an RDD from a Python list
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
rdd = sc.parallelize(numbers)

print(f"Type      : {type(rdd)}")
print(f"Partitions: {rdd.getNumPartitions()}")
print(f"Elements  : {rdd.collect()}")

In [ ]:
# Specify number of partitions and see data distribution
rdd_4p = sc.parallelize(numbers, 4)
print(f"Partitions: {rdd_4p.getNumPartitions()}")
print(f"Data per partition: {rdd_4p.glom().collect()}")

## Step 5 — Creating RDD from S3 File

In [ ]:
# Read orders CSV from S3 as raw text RDD
text_rdd = sc.textFile(f"s3a://{BUCKET}/data/orders.csv")

print(f"Total lines: {text_rdd.count()}")

# Show first 3 lines
for line in text_rdd.take(3):
    print(line)

## Step 6 — Transformations vs Actions

In [ ]:
# Transformations — lazy, nothing runs yet
doubled = rdd.map(lambda x: x * 2)      # transformation
evens   = rdd.filter(lambda x: x % 2 == 0)  # transformation
print("Transformations defined — nothing has run yet")

# Actions — this is when Spark actually executes
print(f"Doubled : {doubled.collect()}")   # action
print(f"Evens   : {evens.collect()}")     # action

## Step 7 — reduceByKey() with Sales Data

In [ ]:
# RDD of (region, sales_amount) tuples
sales_data = [
    ("East", 5000), ("West", 7500), ("South", 3200),
    ("East", 4100), ("West", 6800), ("Midwest", 2900)
]

sales_rdd = sc.parallelize(sales_data)

# Total sales by region
regional_totals = sales_rdd.reduceByKey(lambda a, b: a + b)

# Sort by total descending
sorted_sales = regional_totals.sortBy(lambda x: x[1], ascending=False)

print("Sales by region (descending):")
for region, total in sorted_sales.collect():
    print(f"  {region}: ${total:,}")

---
## Assignment — Day 2

Complete the four tasks below.

---

### Task 1
Using `sc.parallelize()`, create an RDD from this list of order amounts:
`[250, 1500, 89, 3200, 450, 780, 2100, 65, 990, 1750]`

Print the number of partitions and collect all elements.

In [ ]:
# Task 1 — Write your code here


### Task 2
From the RDD in Task 1, use `filter()` to keep only orders above `$500`.
Then use `map()` to apply a 10% discount to each amount.
Print the final result.

In [ ]:
# Task 2 — Write your code here


### Task 3
Create an RDD from this list of `(region, order_count)` tuples:
```python
[("East", 12), ("West", 18), ("East", 9), ("South", 6), 
 ("West", 14), ("Midwest", 7), ("South", 11)]
```
Use `reduceByKey()` to get total order count per region.
Sort by count descending and print the result.

In [ ]:
# Task 3 — Write your code here


### Task 4
Read `customers.csv` from your S3 bucket using `sc.textFile()`.
Print the total line count.
Use `filter()` to remove the header row and print the first 3 data rows using `take()`.

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*